# F2 · When boosting does not win: extrapolation, noisy linear signal, interactions

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 60 minutes · **extended path**

## Objective

Compare Ridge, RandomForest and XGBoost on **five regimes** built to show the strengths and weaknesses of trees: a trend to extrapolate, a linear signal with a lot of noise (with `n = 150` and `n = 5000`), a signal made of interactions (with `n = 600` and `n = 5000`), and the hybrid remedy "Ridge + XGBoost on the residuals".

## Dataset

Five synthetic data generators in `regimes` (given): `trend` (`y = 2t + sin(3t)` with increasing `t`, the test set is the last 30% of the time, so outside the training range), `noisy linear` (30 features of which 6 with nonzero coefficients, noise `σ = 2`) and `interactions` (`y = ±x₁` with sign decided by `x₀`, plus `1.5·[x₂x₃ > 0]`).

## Educational purpose

Know in advance where a tree model (forest or boosting) is a bad choice: **outside the training range it does not extrapolate** (it predicts a constant), with a noisy linear signal and few data it **wastes** its flexibility, while with nonlinear interactions it wins by a lot. And learn the remedy: the linear model captures the global structure, the trees the rest.

## Task

1. Build `table` (test RMSE; index: the five regimes, columns `ridge`, `rf`, `xgb`) with `RidgeCV(alphas=np.logspace(-2, 3, 20))`, `RandomForestRegressor(300, min_samples_leaf=3, random_state=0, n_jobs=-1)` and `XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0, n_jobs=-1)`. Save `winner` (`pd.Series`: method with the lowest RMSE per regime) and `xgb_ridge_ratio` (`xgb / ridge` per regime).
2. **Why the trend?** Redo the `trend` regime with a **random** 70/30 split (interpolation): save `random_trend_table` (`pd.Series` with the three RMSEs) and `random_trend_ratio` (`xgb / ridge`). The problem is the extrapolation, not the function.
3. **Hybrid remedy:** `hybrid = ridge + XGBoost on the residuals` (train Ridge, then `XGBRegressor(...)` on the residuals `y − ridge.predict(X)`; prediction = sum). Save `hybrid_rmse` (`pd.Series`, per regime).
4. Save `rules` (dictionary with three entries: `trend`, `noisy linear`, `interactions` → a short sentence: which model you choose and why).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error


def trend(seed=0):
    rng = np.random.default_rng(seed)
    n = 1200
    t = np.linspace(0, 10, n)
    X = np.c_[t, np.sin(3 * t)]
    y = 2 * t + np.sin(3 * t) + rng.normal(0, 0.5, n)
    c = int(0.7 * n)
    return X[:c], y[:c], X[c:], y[c:]


def noisy_linear(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n + 2000, 30))
    beta = np.zeros(30)
    beta[:6] = [1, -1, 0.8, -0.8, 0.5, 0.5]
    y = X @ beta + rng.normal(0, 2, n + 2000)
    return X[:n], y[:n], X[n:], y[n:]


def interactions(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n + 2000, 8))
    y = np.where(X[:, 0] > 0, X[:, 1], -X[:, 1]) * 2 + (X[:, 2] * X[:, 3] > 0) * 1.5 + rng.normal(0, 0.5, n + 2000)
    return X[:n], y[:n], X[n:], y[n:]


regimes = {
    "trend (extrapolation)": trend(),
    "noisy linear, n=150": noisy_linear(150, 1),
    "noisy linear, n=5000": noisy_linear(5000, 2),
    "interactions, n=600": interactions(600, 3),
    "interactions, n=5000": interactions(5000, 4),
}
models = {
    "ridge": lambda: RidgeCV(alphas=np.logspace(-2, 3, 20)),
    "rf": lambda: RandomForestRegressor(300, min_samples_leaf=3, random_state=0, n_jobs=-1),
    "xgb": lambda: xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0, n_jobs=-1),
}
rmse = lambda y, p: mean_squared_error(y, p, squared=False)

## Your work

In [ ]:
table = pd.DataFrame(index=list(regimes), columns=list(models), dtype=float)
for regime, (X_tr, y_tr, X_te, y_te) in regimes.items():
    ...
table.round(3)

In [ ]:
winner = None
xgb_ridge_ratio = None

In [ ]:
random_trend_table = None
random_trend_ratio = None

In [ ]:
hybrid_rmse = pd.Series(dtype=float)
rules = {}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`models[name]().fit(X_tr, y_tr).predict(X_te)` and `rmse(y_te, p)`. Random split for the trend: `X, y = np.r_[X_tr, X_te], np.r_[y_tr, y_te]` then `train_test_split(X, y, test_size=0.3, random_state=0)`. Hybrid: `r = RidgeCV(...).fit(X_tr, y_tr)`, `g = xgb.XGBRegressor(...).fit(X_tr, y_tr - r.predict(X_tr))`, prediction `r.predict(X_te) + g.predict(X_te)`.'):
    check(table.notna().all().all() and table.shape == (5, 3), "complete 5 x 3 table", "RMSE per regime and method")
    check(winner is not None and winner["trend (extrapolation)"] == "ridge" and winner["noisy linear, n=150"] == "ridge" and winner["noisy linear, n=5000"] == "ridge", "Ridge wins on the trend and on the noisy linear signal (even with 5000 examples)", "idxmin per row")
    check(winner is not None and winner["interactions, n=600"] in ("rf", "xgb") and winner["interactions, n=5000"] in ("rf", "xgb") and table.loc["interactions, n=5000", "ridge"] > 2.5 * table.loc["interactions, n=5000", "xgb"], "with interactions the trees win, and Ridge has an RMSE over 2.5 times larger at n=5000", "ridge does not see the interactions")
    check(xgb_ridge_ratio is not None and xgb_ridge_ratio["trend (extrapolation)"] > 4 and 1.0 < xgb_ridge_ratio["noisy linear, n=5000"] < 1.15 and 1.02 < xgb_ridge_ratio["noisy linear, n=150"], "on the trend XGBoost has an RMSE over 4 times larger; on the noisy linear one it is always worse (slightly at n=5000, less than 15%)", "xgb / ridge")
    check(random_trend_table is not None and random_trend_ratio < 1.2 and random_trend_table.min() < 0.6, "with a random split the three methods are close (xgb / ridge below 1.2): the disaster was the extrapolation", "random 70/30 split of the trend regime")
    check(len(hybrid_rmse) == 5 and all(hybrid_rmse[r] <= 1.1 * table.loc[r].min() for r in table.index), "the hybrid is within 10% of the best in all the regimes", "ridge + xgb on the residuals")
    check(hybrid_rmse["trend (extrapolation)"] < 0.7 and hybrid_rmse["interactions, n=5000"] < 0.85, "the hybrid solves the trend (below 0.7) and the interactions at n=5000 (below 0.85)", "the trees correct only the residuals")
    check(set(rules) == {"trend", "noisy linear", "interactions"} and all(isinstance(v, str) and len(v) > 15 for v in rules.values()), "three rules, one per regime", "one sentence per regime")
done()

---
The commented solution is in `solutions/F_compare/F2_when_boosting_does_not_win.ipynb`. Open it after you pass the check, or after you have really tried.